In [1]:
import ipykernel, sys
print("ipykernel:", ipykernel.__version__)


ipykernel: 7.1.0


In [2]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from utils import train_model, test_model
from utils import H5VideoDataset, pad_collate_fn, compute_pos_weight_and_prior
from utils import plot_ethogram_pair, make_bucket_batch_sampler_for_subset
from utils import build_weighted_sampler
import h5py

In [3]:
# Paths
h5_path = r"D:\Mouse_behavior_data\D21\input_output_data_downsample_444.h5"
model_save_path = 'model.pth'

# Parameters
batch_size = 2
num_epochs = 25

merge_group = [[4, 6], [5, 7]]
drop_indices = [2, 3, 8]

In [4]:
dataset = H5VideoDataset(h5_path, merge_groups=merge_group, drop_indices=drop_indices)

# Infer number of labels from first sample
_, y0, *_ = dataset[0]  # y0 is (T, C)
num_labels = y0.shape[1]
print(f"Detected number of labels: {num_labels}")

# --- Ratios ---
val_ratio  = 0.15   # 15% validation
test_ratio = 0.15   # 15% test
assert 0 < val_ratio < 1 and 0 < test_ratio < 1 and (val_ratio + test_ratio) < 1

N = len(dataset)
val_size  = max(1, int(N * val_ratio))
test_size = max(1, int(N * test_ratio))
train_size = N - val_size - test_size
assert train_size > 0, "Not enough samples for the requested splits."

g = torch.Generator().manual_seed(42)
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size], generator=g)

train_batch_sampler = make_bucket_batch_sampler_for_subset(train_set, dataset.lengths, batch_size, bucket_size=min(len(train_set), 16), shuffle=True)
val_batch_sampler   = make_bucket_batch_sampler_for_subset(val_set,   dataset.lengths, batch_size, bucket_size=len(val_set),            shuffle=False)
test_batch_sampler  = make_bucket_batch_sampler_for_subset(test_set,  dataset.lengths, batch_size, bucket_size=len(test_set),           shuffle=False)

stats_loader = DataLoader(
    train_set, 
    batch_sampler=train_batch_sampler,
    collate_fn=lambda b: pad_collate_fn(
        b, training=False, max_T=None, positive_bias=False
    ),
    num_workers=0, pin_memory=torch.cuda.is_available()
)

# train_sampler = build_weighted_sampler(train_set)
train_loader = DataLoader(
    train_set, batch_sampler=train_batch_sampler,
    collate_fn=lambda b: pad_collate_fn(b, training=True, max_T=200, positive_bias=True),
    num_workers=0, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_set, batch_sampler=val_batch_sampler,
    collate_fn=lambda b: pad_collate_fn(b, training=False, max_T=None, positive_bias=False),
    num_workers=0, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_set, batch_sampler=test_batch_sampler,
    collate_fn=lambda b: pad_collate_fn(b, training=False, max_T=None, positive_bias=False),
    num_workers=0, pin_memory=torch.cuda.is_available()
)

Detected number of labels: 4


In [5]:
# Train
train_model(
    train_loader=train_loader,
    num_labels=num_labels,
    num_epochs=num_epochs,
    save_path=model_save_path,
    use_asl=True, asl_gamma_neg=2.0, asl_gamma_pos=0.0, asl_clip=0.05,
    lr=1e-4,
    stats_loader=stats_loader,
    # New parameters for text encoding (enable if you have stim_sentence data)
    use_text_encoder=True,  # Set to True to use sentence transformer for stimulus metadata
    text_mode="sentence_transformer",  # "sentence_transformer" to embed raw text; "precomputed" for pre-embedded vectors
    text_in_dim=None,  # Set to embedding dim (e.g., 384) if text_mode="precomputed"
    text_model_name="all-MiniLM-L6-v2"  # Set to model name (e.g., "all-MiniLM-L6-v2") if text_mode="sentence_transformer"
)

[STATS] prior (unbiased): [0.01073  0.011588 0.017811 0.01588 ]
[STATS] pos_weight (unbiased): [10. 10. 10. 10.]
[Epoch 1] Train Loss: 0.0950
[Epoch 2] Train Loss: 0.1128
[Epoch 3] Train Loss: 0.1294
[Epoch 4] Train Loss: 0.1248
[Epoch 5] Train Loss: 0.1172
[Epoch 6] Train Loss: 0.1104
[Epoch 7] Train Loss: 0.1058
[Epoch 8] Train Loss: 0.0964
[Epoch 9] Train Loss: 0.0918
[Epoch 10] Train Loss: 0.0843
[Epoch 11] Train Loss: 0.0758
[Epoch 12] Train Loss: 0.0696
[Epoch 13] Train Loss: 0.0601
[Epoch 14] Train Loss: 0.0538
[Epoch 15] Train Loss: 0.0507
[Epoch 16] Train Loss: 0.0441
[Epoch 17] Train Loss: 0.0384
[Epoch 18] Train Loss: 0.0376
[Epoch 19] Train Loss: 0.0336
[Epoch 20] Train Loss: 0.0328
[Epoch 21] Train Loss: 0.0331
[Epoch 22] Train Loss: 0.0285
[Epoch 23] Train Loss: 0.0267
[Epoch 24] Train Loss: 0.0255
[Epoch 25] Train Loss: 0.0254
Model saved to: model.pth


BehaviorPredictor(
  (cnn): Sequential(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (scale_activati

In [6]:
# Test
th_vec = test_model(
    test_loader=test_loader, 
    model_path=model_save_path, 
    num_labels=num_labels,
    threshold=0.4, 
    use_hysteresis=False, 
    t_high=0.45, 
    t_low=0.30, 
    min_len=3, 
    presence_threshold=0.25,
    # New parameters for text encoding (must match training setup)
    use_text_encoder=True,  # Match train_model setting
    text_mode="sentence_transformer",  # Match train_model setting
    text_in_dim=None,  # Match train_model setting
    text_model_name="all-MiniLM-L6-v2"  # Match train_model setting
)

print(f"Per-class thresholds: {th_vec}")


Masked per-frame, per-label accuracy: 0.9933
Micro Precision: 0.4000 | Micro Recall: 0.1053 | Micro F1: 0.1667
[QUAL] Presence(probs≥0.25) — Acc: 0.5625 | Prec: 0.5714 | Rec: 0.5000 | F1: 0.5333
[QUAL] Ethogram exact-match accuracy: 0.2500 (over 4 clips)
[QUAL] Ethogram set IoU (avg over clips with non-empty union): 0.3333 (n=4)
[QUAL] Per-class presence recall: [0.5, 0.0, 0.6669999957084656, 0.5]
[QUAL] Per-class presence support (GT #clips): [2, 1, 3, 2]
Per-class thresholds: None
